# V19: Portfolio Strategy Comparison with HMM & AI Integration

This notebook provides a comprehensive backtest of various portfolio management strategies, now including **Hidden Markov Models (HMM)** as a market regime filter. 

Strategies included:
1. **Markowitz Only**: Standard optimization without market timing.
2. **AI-Gated Markowitz**: Risk-On/Off based on Random Forest predictions.
3. **HMM-Gated Markowitz**: Risk-On/Off based on Hidden Markov Model states.
4. **Hierarchical Gated MK**: Combines AI and HMM for a double-layer safety gate.
5. **Full Enhanced (AI+HMM+Graph+MK)**: Uses the Hierarchical Gate + Graph Filter (MIS) + Markowitz.
6. **BTC Benchmark**: Buy and hold Bitcoin.

In [3]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from scipy.optimize import minimize
from sklearn.ensemble import RandomForestClassifier
from hmmlearn import hmm
import warnings
warnings.filterwarnings('ignore')

# --- 1. Load Data ---
file_path = '../dataset_2023_2025.xlsx'
if not os.path.exists(file_path):
    file_path = 'dataset_2023_2025.xlsx'

data = pd.read_excel(file_path, index_col=0, parse_dates=True)
returns = data.pct_change().dropna()
market_return = returns.mean(axis=1)

print(f"✓ Data Loaded: {len(data)} rows")

✓ Data Loaded: 990 rows


In [4]:
# --- 2. HMM Training (Global Regime Layer) ---
print("Training Global HMM...")
market_log_ret = np.log1p(market_return).replace([np.inf, -np.inf], np.nan).dropna()
X_global = market_log_ret.values.reshape(-1, 1)

global_hmm = hmm.GaussianHMM(n_components=2, covariance_type="full", n_iter=1000, random_state=42)
global_hmm.fit(X_global)

# Identify Bull State (Higher mean return)
bull_state = np.argmax([global_hmm.means_[i][0] for i in range(2)])
hmm_regimes = pd.Series(global_hmm.predict(X_global) == bull_state, index=market_log_ret.index).astype(int)

print(f"✓ Global HMM Trained. Bull State index: {bull_state}")

Training Global HMM...
✓ Global HMM Trained. Bull State index: 0


In [5]:
# --- 3. AI Gatekeeper Training (Random Forest) ---
print("Training AI Gatekeeper...")
features = pd.DataFrame(index=returns.index)
features['Vol_20'] = market_return.rolling(window=20).std()
features['Mom_20'] = market_return.rolling(window=20).mean()
features['Mom_50'] = market_return.rolling(window=50).mean()
features['Target'] = (market_return.shift(-1) > 0).astype(int)
features = features.dropna()

train_mask = (features.index.year >= 2023) & (features.index.year <= 2024)
test_mask = (features.index.year == 2025)

X_train, y_train = features.loc[train_mask, ['Vol_20', 'Mom_20', 'Mom_50']], features.loc[train_mask, 'Target']
X_test, y_test = features.loc[test_mask, ['Vol_20', 'Mom_20', 'Mom_50']], features.loc[test_mask, 'Target']

rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)
ai_probs = pd.Series(rf_model.predict_proba(X_test)[:, 1], index=X_test.index)

print("✓ AI Gatekeeper Trained.")

Training AI Gatekeeper...
✓ AI Gatekeeper Trained.


In [6]:
# --- 4. Strategy Core Logic ---

def get_mis_assets(returns_window, correlation_threshold=0.5):
    corr_mat = returns_window.corr()
    G = nx.Graph()
    assets = returns_window.columns
    G.add_nodes_from(assets)
    for i in range(len(assets)):
        for j in range(i+1, len(assets)):
            if corr_mat.iloc[i, j] > correlation_threshold:
                G.add_edge(assets[i], assets[j])
    mis = nx.approximation.maximum_independent_set(G)
    return list(mis)

def optimize_markowitz(selected_returns):
    if len(selected_returns.columns) == 0: return {}
    if len(selected_returns.columns) == 1: return {selected_returns.columns[0]: 1.0}
    
    mu = selected_returns.mean() * 252
    sigma = selected_returns.cov() * 252
    
    def neg_sharpe_ratio(weights):
        port_ret = np.sum(weights * mu)
        port_vol = np.sqrt(np.dot(weights.T, np.dot(sigma, weights)))
        return -port_ret / port_vol if port_vol > 0 else 0
    
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for _ in range(len(mu)))
    guess = [1. / len(mu)] * len(mu)
    
    try:
        res = minimize(neg_sharpe_ratio, guess, method='SLSQP', bounds=bounds, constraints=constraints)
        return dict(zip(selected_returns.columns, res.x))
    except:
        return dict(zip(selected_returns.columns, guess))

def run_simulation(strategy_type, test_dates, returns, ai_probs, hmm_regimes, lookback=30):
    current_value = 10000.0
    history = [current_value]
    daily_returns = [0.0]
    dates = [test_dates[0]]
    
    diag_regimes = []
    diag_n_assets = []
    diag_weights = []
    
    for i, date in enumerate(test_dates[:-1]):
        # 1. Gate Logic
        ai_gate = ai_probs.loc[date] > 0.5
        hmm_gate = hmm_regimes.reindex(test_dates, method='ffill').loc[date] == 1
        
        if strategy_type == 'Markowitz Only':
            is_bull = True
        elif strategy_type == 'AI-Gated Markowitz':
            is_bull = ai_gate
        elif strategy_type == 'HMM-Gated Markowitz':
            is_bull = hmm_gate
        elif strategy_type == 'Hierarchical MK':
            is_bull = ai_gate and hmm_gate
        elif strategy_type == 'Full Enhanced':
            is_bull = ai_gate and hmm_gate
        else:
            is_bull = True
            
        diag_regimes.append('BULL' if is_bull else 'BEAR')
        
        loc_idx = returns.index.get_loc(date)
        window_rets = returns.iloc[loc_idx-lookback:loc_idx]
        
        if not is_bull:
            weights = {'CASH': 1.0}
        else:
            if strategy_type == 'Full Enhanced':
                selected = get_mis_assets(window_rets)
                weights = optimize_markowitz(window_rets[selected])
            else:
                weights = optimize_markowitz(window_rets)
        
        diag_n_assets.append(len([w for w in weights if w != 'CASH']))
        diag_weights.append(weights)
        
        # 2. Rebalancing & returns
        next_date = test_dates[i+1]
        next_rets = returns.loc[next_date]
        
        day_ret = 0 if 'CASH' in weights else sum(w * next_rets[a] for a, w in weights.items())
        
        current_value *= (1 + day_ret)
        history.append(current_value)
        daily_returns.append(day_ret)
        dates.append(next_date)
        
    df_res = pd.DataFrame({
        'Date': dates,
        'Portfolio_Value': history,
        'Daily_Return': daily_returns,
        'Status': diag_regimes + [np.nan],
        'Assets_Count': diag_n_assets + [0]
    }).set_index('Date')
    
    weights_df = pd.DataFrame(diag_weights + [{}], index=dates)
    df_res = pd.concat([df_res, weights_df], axis=1)
    
    return df_res

In [7]:
# --- 5. Run Simulations & Export to Excel ---
test_dates = ai_probs.index
output_file = "v19_portfolio_backtest_2025.xlsx"

print(f"Backtesting for {len(test_dates)} days...")

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    strategies = [
        ('Markowitz Only', 'Baseline'), 
        ('AI-Gated Markowitz', 'AI Gated'),
        ('HMM-Gated Markowitz', 'HMM Gated'),
        ('Hierarchical MK', 'AI+HMM Gated'),
        ('Full Enhanced', 'AI+HMM+Graph+MK')
    ]
    
    for strat_name, sheet_name in strategies:
        print(f"- Processing {strat_name}...")
        df = run_simulation(strat_name, test_dates, returns, ai_probs, hmm_regimes)
        df.to_excel(writer, sheet_name=sheet_name)
        
    # BTC Benchmark
    print("- Processing BTC Benchmark...")
    btc_price = data.loc[test_dates, 'BTC-USD']
    df_btc = pd.DataFrame({
        'Date': test_dates,
        'Portfolio_Value': (btc_price / btc_price.iloc[0] * 10000),
        'Daily_Return': btc_price.pct_change().fillna(0),
        'BTC_Weight': 1.0
    }).set_index('Date')
    df_btc.to_excel(writer, sheet_name='BTC Benchmark')

print(f"\n✅ ALL SUCCESS: Exported to {output_file}")

Backtesting for 365 days...
- Processing Markowitz Only...
- Processing AI-Gated Markowitz...
- Processing HMM-Gated Markowitz...
- Processing Hierarchical MK...
- Processing Full Enhanced...
- Processing BTC Benchmark...

✅ ALL SUCCESS: Exported to v19_portfolio_backtest_2025.xlsx
